In [ ]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

In [ ]:
# ============================================
# PHASE 2: DISTANCE/TIME MATRIX FOR AED COVERAGE
# UP Mindanao Campus
# ============================================

# Data from your file (21 nodes, 12 candidates marked)
data = {
    'node': ['atrium', 'mosque', 'rcdg', 'kalimudan', 'soccer_field', 'ebl',
             'library', 'carim', 'dorm2', 'csm', 'diwa', 'himati_house',
             'usc_house', 'castillo', 'ate_lings', 'lydia', 'aquatics',
             'dhk_training_gym', 'dc_sports_complex', 'cultural_center', 'som'],
    'latitude': [7.085684249, 7.086108865, 7.086787069, 7.084928961, 7.084748209,
                 7.084045294, 7.084869363, 7.084588790, 7.084203983, 7.083493536,
                 7.083814642, 7.084490747, 7.084870379, 7.084132736, 7.084104305,
                 7.084340009, 7.083787884, 7.086690736, 7.085421685, 7.086800055,
                 7.087098075],
    'longitude': [125.484813939, 125.487095984, 125.485253708, 125.486917759, 125.486033068,
                  125.485191200, 125.483687583, 125.479256671, 125.477651291, 125.477314120,
                  125.476855733, 125.486483181, 125.486525312, 125.485538144, 125.486143151,
                  125.477727886, 125.471692288, 125.471794188, 125.470452591, 125.481304364,
                  125.484034039],
    'is_candidate': [True, False, True, False, False, True, True, True, False, True, False,
                     False, False, False, False, False, True, True, True, True, True],
    'activity_type': ['sedentary', 'sedentary', 'vigorous', 'moderate', 'vigorous', 'moderate',
                      'sedentary', 'sedentary', 'moderate', 'sedentary', 'sedentary', 'sedentary',
                      'sedentary', 'moderate', 'sedentary', 'moderate', 'vigorous', 'vigorous',
                      'vigorous', 'sedentary', 'sedentary'],
    'est_peak_pop': [350, 50, 350, 50, 50, 250, 100, 200, 10, 350, 50, 50, 50, 50, 50, 50, 50, 350, 100, 50, 350]
}

df = pd.DataFrame(data)

In [ ]:
# Parameters
WALKING_SPEED = 1.4  # m/s
CORRECTION_FACTOR = 1.35  # hilly campus adjustment
COVERAGE_THRESHOLD = 300  # seconds (5 minutes)

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate straight-line distance in meters between two lat/lon points"""
    R = 6371000  # Earth's radius in meters

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c

def walking_time(lat1, lon1, lat2, lon2):
    """Estimate walking time in seconds (straight-line × correction ÷ speed)"""
    straight_dist = haversine_distance(lat1, lon1, lat2, lon2)
    adjusted_dist = straight_dist * CORRECTION_FACTOR
    time_sec = adjusted_dist / WALKING_SPEED
    return time_sec

# Identify AED candidates and population nodes
aed_candidates = df[df['is_candidate'] == True].copy()
population_nodes = df.copy()

# Build matrix
matrix_data = []
for _, aed in aed_candidates.iterrows():
    row = {'AED_Candidate': aed['node']}
    for _, node in population_nodes.iterrows():
        time_sec = walking_time(aed['latitude'], aed['longitude'],
                                node['latitude'], node['longitude'])
        row[node['node']] = round(time_sec)
    matrix_data.append(row)

# Create DataFrame
time_matrix = pd.DataFrame(matrix_data)
time_matrix.set_index('AED_Candidate', inplace=True)

# Add coverage status (True if ≤300 seconds)
coverage_matrix = time_matrix.map(lambda x: x <= COVERAGE_THRESHOLD)

In [ ]:
print("=" * 160)
print(f"WALKING TIME MATRIX (seconds) - Correction Factor: {CORRECTION_FACTOR}, Speed: {WALKING_SPEED} m/s")
print(f"✅ = covered within {COVERAGE_THRESHOLD//60} minutes (≤{COVERAGE_THRESHOLD} sec), ❌ = not covered")
print("=" * 160)
print()

formatted_data = []
for aed in time_matrix.index:
    row = {'AED_Candidate': aed}
    for node in time_matrix.columns:
        time_val = time_matrix.loc[aed, node]
        if time_val <= COVERAGE_THRESHOLD:
            row[node] = f"{time_val}✅"
        else:
            row[node] = f"{time_val}❌"
    formatted_data.append(row)

formatted_matrix = pd.DataFrame(formatted_data)
formatted_matrix.set_index('AED_Candidate', inplace=True)

print(formatted_matrix.to_string())
print()
print("=" * 160)

WALKING TIME MATRIX (seconds) - Correction Factor: 1.35, Speed: 1.4 m/s
✅ = covered within 5 minutes (≤300 sec), ❌ = not covered

                  atrium mosque   rcdg kalimudan soccer_field    ebl library carim dorm2   csm  diwa himati_house usc_house castillo ate_lings lydia aquatics dhk_training_gym dc_sports_complex cultural_center    som
AED_Candidate                                                                                                                                                                                                          
atrium                0✅   247✅   127✅      238✅         164✅   180✅    148✅  603❌  778❌  832❌  870❌         219✅      202✅     183✅      221✅  768❌    1411❌            1390❌             1528❌            392❌   173✅
rcdg                127✅   209✅     0✅      267✅         234✅   294✅    265✅  680❌  855❌  916❌  949❌         279✅      246✅     286✅      303❌  843❌    1478❌            1432❌             1582❌            420❌   134✅
ebl   

In [ ]:
# ============================================
# SUMMARY: Nodes covered per AED
# ============================================

coverage_summary = pd.DataFrame({
    'AED_Candidate': time_matrix.index,
    'Nodes_Covered': coverage_matrix.sum(axis=1).values,
    'Coverage_Percent': (coverage_matrix.sum(axis=1) / len(population_nodes) * 100).values,
    'Uncovered_Nodes': [', '.join([col for col in coverage_matrix.columns if not coverage_matrix.loc[aed, col]])
                        for aed in coverage_matrix.index]
})

print("\nCOVERAGE SUMMARY (per AED candidate)")
print("-" * 80)
for _, row in coverage_summary.iterrows():
    print(f"{row['AED_Candidate']:20} → {row['Nodes_Covered']:2}/{len(population_nodes)} nodes covered ({row['Coverage_Percent']:.0f}%)")
    if row['Uncovered_Nodes']:
        print(f"{'':20}   Uncovered: {row['Uncovered_Nodes']}")
    else:
        print(f"{'':20}   All nodes covered!")


COVERAGE SUMMARY (per AED candidate)
--------------------------------------------------------------------------------
atrium               → 12/21 nodes covered (57%)
                       Uncovered: carim, dorm2, csm, diwa, lydia, aquatics, dhk_training_gym, dc_sports_complex, cultural_center
rcdg                 → 11/21 nodes covered (52%)
                       Uncovered: carim, dorm2, csm, diwa, ate_lings, lydia, aquatics, dhk_training_gym, dc_sports_complex, cultural_center
ebl                  → 11/21 nodes covered (52%)
                       Uncovered: carim, dorm2, csm, diwa, lydia, aquatics, dhk_training_gym, dc_sports_complex, cultural_center, som
library              →  9/21 nodes covered (43%)
                       Uncovered: mosque, kalimudan, carim, dorm2, csm, diwa, usc_house, lydia, aquatics, dhk_training_gym, dc_sports_complex, cultural_center
carim                →  5/21 nodes covered (24%)
                       Uncovered: atrium, mosque, rcdg, kalimudan, soccer_

In [ ]:
# ============================================
# RAW NUMERICAL MATRIX
# ============================================

print("\n" + "=" * 160)
print("RAW NUMERICAL MATRIX (seconds)")
print("=" * 160)
print(time_matrix.to_string())


RAW NUMERICAL MATRIX (seconds)
                   atrium  mosque  rcdg  kalimudan  soccer_field   ebl  library  carim  dorm2  csm  diwa  himati_house  usc_house  castillo  ate_lings  lydia  aquatics  dhk_training_gym  dc_sports_complex  cultural_center   som
AED_Candidate                                                                                                                                                                                                                      
atrium                  0     247   127        238           164   180      148    603    778  832   870           219        202       183        221    768      1411              1390               1528              392   173
rcdg                  127     209     0        267           234   294      265    680    855  916   949           279        246       286        303    843      1478              1432               1582              420   134
ebl                   180     300   294        207      

In [ ]:
# ============================================
# OPTIONAL: Export to Excel
# ============================================

# Uncomment to export to Excel file
# with pd.ExcelWriter('upmin_aed_coverage_matrix.xlsx') as writer:
#     time_matrix.to_excel(writer, sheet_name='Raw_Time_Matrix')
#     formatted_matrix.to_excel(writer, sheet_name='Formatted_Matrix')
#     coverage_summary.to_excel(writer, sheet_name='Coverage_Summary', index=False)
#     df.to_excel(writer, sheet_name='Node_Data', index=False)
# print("\n✅ Exported to 'upmin_aed_coverage_matrix.xlsx'")

In [ ]:
# ============================================
# ADDITIONAL ANALYSIS: Priority by population
# ============================================

print("\n" + "=" * 160)
print("PRIORITY ANALYSIS (Weighted by Population)")
print("=" * 160)

# Calculate weighted coverage score (population × coverage)
weighted_scores = []
for aed in aed_candidates['node']:
    score = 0
    for _, node in population_nodes.iterrows():
        if coverage_matrix.loc[aed, node['node']]:
            score += node['est_peak_pop']
    weighted_scores.append(score)

priority_df = pd.DataFrame({
    'AED_Candidate': aed_candidates['node'].values,
    'Population_Covered': weighted_scores,
    'Total_Population': population_nodes['est_peak_pop'].sum(),
    'Population_Percent': [s/population_nodes['est_peak_pop'].sum()*100 for s in weighted_scores]
}).sort_values('Population_Covered', ascending=False)

print("\nTop AED candidates by population coverage:")
print(priority_df.to_string(index=False))


PRIORITY ANALYSIS (Weighted by Population)

Top AED candidates by population coverage:
    AED_Candidate  Population_Covered  Total_Population  Population_Percent
           atrium                1750              2960           59.121622
             rcdg                1700              2960           57.432432
          library                1600              2960           54.054054
              ebl                1400              2960           47.297297
              som                1200              2960           40.540541
              csm                 660              2960           22.297297
            carim                 660              2960           22.297297
dc_sports_complex                 500              2960           16.891892
 dhk_training_gym                 450              2960           15.202703
  cultural_center                 400              2960           13.513514
         aquatics                 150              2960            5.067568


In [ ]:
# ============================================
# VIGOROUS ACTIVITY ANALYSIS (higher risk areas)
# ============================================

print("\n" + "=" * 160)
print("VIGOROUS ACTIVITY NODES COVERAGE (higher cardiac risk)")
print("=" * 160)

vigorous_nodes = population_nodes[population_nodes['activity_type'] == 'vigorous']['node'].tolist()
print(f"\nVigorous activity nodes: {', '.join(vigorous_nodes)}")

vigorous_coverage = pd.DataFrame({
    'AED_Candidate': time_matrix.index,
    'Vigorous_Nodes_Covered': [sum([coverage_matrix.loc[aed, node] for node in vigorous_nodes]) for aed in time_matrix.index],
    'Total_Vigorous_Nodes': len(vigorous_nodes)
})

for _, row in vigorous_coverage.iterrows():
    print(f"{row['AED_Candidate']:20} → {row['Vigorous_Nodes_Covered']}/{row['Total_Vigorous_Nodes']} vigorous nodes covered")

print("\n" + "=" * 160)
print("✅ Analysis complete!")


VIGOROUS ACTIVITY NODES COVERAGE (higher cardiac risk)

Vigorous activity nodes: rcdg, soccer_field, aquatics, dhk_training_gym, dc_sports_complex
atrium               → 2/5 vigorous nodes covered
rcdg                 → 2/5 vigorous nodes covered
ebl                  → 2/5 vigorous nodes covered
library              → 2/5 vigorous nodes covered
carim                → 0/5 vigorous nodes covered
csm                  → 0/5 vigorous nodes covered
aquatics             → 2/5 vigorous nodes covered
dhk_training_gym     → 2/5 vigorous nodes covered
dc_sports_complex    → 3/5 vigorous nodes covered
cultural_center      → 0/5 vigorous nodes covered
som                  → 1/5 vigorous nodes covered

✅ Analysis complete!
